# CS 691 · Module 15 · Explaining and testing a hybrid decision

Trace a screening decision, compare a learned model and a rule on the same held-out requests, and locate an error using independent evidence.

**Before you begin.** You have used scores, rules and evidence records to justify a decision. You have also seen how several explanations can fit the same observations.

A desk forwards a request when the photo score is at least 0.80 and the camera is available. The score is 0.93 and the inventory says unavailable. What would the rule do?

<details><summary>Reveal the answer</summary>

It would hold the request. Both conditions must hold, and the inventory condition fails even though the score passes.

</details>

## What made this request reach the clerk?

Ava submits a photo of the item she wants to borrow. A learned model gives a score for whether the photo matches camera C1. Our fictional desk sends the request to a clerk if the score is at least 0.80 and the inventory lists C1 as available. The clerk makes the final decision.

An explanation can retrace this decision using its actual inputs. Record the photo ID, model score, inventory-row ID and the result of the two tests. A clerk can then inspect either source.

**Predict before running the cell.** Would “the model rejected the photo” explain why this request was held?

In [ ]:
photo, score = "P7", 0.91
inventory, available = "I3", False
match = score >= 0.80
print(photo, "passes cutoff:", match)
print(inventory, "available:", available)
print("send to clerk:", match and available)

<details><summary>Reveal output and explanation</summary>

```text
P7 passes cutoff: True
I3 available: False
send to clerk: False
```

No. The photo passed its cutoff. Inventory row I3 marked C1 unavailable, which made the final test fail. This trace explains the system’s behavior even if I3 later turns out to contain an error.

</details>

## Give the test a reference answer

Choose the decision you want to evaluate. Here a reviewer independently checks whether the photographed item really was C1 and whether C1 actually was available at the request time. Only a true match and actual availability together warrant a send. A held-out set consists of requests kept out of training and choices such as the cutoff. Compare every method on these same requests.

The six reviewed decisions run from R1 to R6. True means the request met both conditions. A baseline is a simple comparison method. One baseline holds every request. Accuracy counts agreements and divides by all requests.

**Predict before running the cell.** Can the desk claim that this baseline successfully sends eligible requests to the clerk?

In [ ]:
reviewed = [True, False, False, True, False, False]
always_hold = [False] * len(reviewed)
correct = sum(pred == ref for pred, ref in zip(always_hold, reviewed))
print(correct, "of", len(reviewed))

<details><summary>Reveal output and explanation</summary>

```text
4 of 6
```

It earns four correct holds but sends none of the two eligible requests. Overall accuracy hides that failure.

</details>

## Which part of the system helped?

Test a score-only decision and a decision that also uses inventory. Keep the same six requests, the same reference decisions and the same cutoff. Removing a component while keeping the rest fixed is an ablation. The comparison shows what that component contributes on these cases.

Each row lists the photo score, recorded availability and reviewer’s decision. The score-only method sends any row at or above 0.80. The hybrid method also requires availability.

**Predict before running the cell.** Which row changes when inventory is added, and do the four hybrid agreements settle which system is best?

In [ ]:
rows = [(0.95, True, True), (0.90, False, False),
        (0.40, True, False), (0.60, True, True),
        (0.85, True, False), (0.35, False, False)]
score_only = hybrid = 0
for score, available, reference in rows:
    match = score >= 0.80
    score_only += (match == reference)
    hybrid += ((match and available) == reference)
print(score_only, hybrid)

<details><summary>Reveal output and explanation</summary>

```text
3 4
```

Only R2 changes, from an incorrect send to a correct hold. Hybrid and always-hold both agree four times. The hybrid sends eligible R1 and ineligible R5, so the choice also depends on the costs of a missed request and an unnecessary review.

</details>

## Diagnose the two kinds of mistakes

A false positive is a request the system sends that the reviewer says should be held. A false negative is an eligible request the system holds. Precision asks what fraction of sent requests were eligible. Recall asks what fraction of eligible requests were sent. To locate a mistake, inspect independent evidence for each step.

The hybrid sends R1 and R5. The reviewer marks R1 and R4 eligible. The reviewer also confirms availability in R4 and R5, so their errors warrant inspection of the photo decisions.

**Predict before running the cell.** What would you inspect before raising the cutoff to remove R5?

In [ ]:
sent = {"R1", "R5"}
eligible = {"R1", "R4"}
correct_sends = sent & eligible
print("unnecessary review:", sorted(sent - eligible))
print("missed:", sorted(eligible - sent))
print("precision:", len(correct_sends) / len(sent))
print("recall:", len(correct_sends) / len(eligible))

<details><summary>Reveal output and explanation</summary>

```text
unnecessary review: ['R5']
missed: ['R4']
precision: 0.5
recall: 0.5
```

Inspect the photos, their reference labels and their scores. R5 is an incorrect match at 0.85, but R4 is a real match at only 0.60. Raising the cutoff cannot recover R4. Choose changes on development cases, then evaluate on fresh held-out cases.

</details>

## Put each send in the right comparison set

The ampersand keeps IDs found in both sets, and the minus sign keeps IDs found only in the set on its left. Counting correct sends and dividing by all sends gives precision. Dividing the same count by all eligible requests gives recall.

Replace this line:
sent = {"R1", "R5"}
with:
sent = {"R1", "R4", "R5"}
What will print?

In [ ]:
sent = {"R1", "R5"}
eligible = {"R1", "R4"}
correct_sends = sent & eligible
print("unnecessary review:", sorted(sent - eligible))
print("missed:", sorted(eligible - sent))
print("precision:", len(correct_sends) / len(sent))
print("recall:", len(correct_sends) / len(eligible))

<details><summary>Worked answer after your change</summary>

unnecessary review: ['R5']
missed: []
precision: 0.6666666666666666
recall: 1.0

R1 and R4 are both caught, so recall reaches 2/2. R5 is still an unnecessary review, which leaves precision at 2/3.

</details>

## Exercise 1

On ten reviewed requests, eight should be held. How accurate is always-hold, and what is its recall for eligible requests?

<details><summary>Hint</summary>

Count correct holds, then count eligible requests sent.

</details>

<details><summary>Worked answer</summary>

It agrees eight times out of ten, or 80%. It sends zero of the two eligible requests, giving 0% recall. Precision for sends has a zero denominator here.

</details>

## Exercise 2

The hybrid and the score-only method give the same wrong answer for R20. What evidence could distinguish a bad photo score from a bad inventory record?

<details><summary>Hint</summary>

Use independent information available for the same request and time.

</details>

<details><summary>Worked answer</summary>

Compare the photo with the correct item and obtain an independently checked inventory snapshot at the request time. Replay both tests with those values. The failed comparison locates a possible error, and both components could be wrong.

</details>

## Exercise 3

A team tries 20 score cutoffs on its final test set and reports the one that sent the most eligible requests with few incorrect sends. How would you redesign that evaluation?

<details><summary>Hint</summary>

Separate the examples used to choose the cutoff from the examples used to report performance.

</details>

<details><summary>Worked answer</summary>

Choose the cutoff on a development set. Fix the threshold, inventory rule and evaluation measures, then run the comparison once on new held-out requests. Keep repeated copies of the same request together when dividing the data.

</details>

In the capstone, a hospital teaching-room clerk needs to review equipment handoffs. You will combine the score, evidence lookup and time rule in one auditable decision, then test its weak spots.

A teaching-room audit will screen staff notes and compare a check time with a handoff time. If one ticket is labeled incorrectly, what should you save so someone can tell which step went wrong?

<details><summary>Reveal the answer</summary>

Keep the ticket ID, original note and its score, the handoff record, both event times and the rule used. A reviewer can then test whether the note was interpreted correctly and replay the time comparison independently.

</details>